In [1]:
!pip install sqlalchemy
!pip install ipython-sql
!pip install pymysql

In [2]:
import sqlite3
from sqlalchemy import create_engine ## Create a connection to the SQLite database
import pandas as pd


In [22]:
# connect sqlite database (*it will create the database if it doesn't exist)
# to start up the database, we need to create an engine, which is a common interface to the database.
# now lets start up the environment and create a connection to the database

%reload_ext sql
engine = create_engine('sqlite:///bonga.db')
%config sql.conn_name = 'engine'

conn = sqlite3.connect('bonga.db')
cursor = conn.cursor()

# now we can use the connection to execute SQL commands. For example,
# we can create a table called 'users' with columns 'id', 'name', and 'email'.

%sql sqlite:///bonga.db

# create a table now

cursor.execute('''
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
               price DECIMAL(10, 2) NOT NULL,
               category VARCHAR(100) NOT NULL
               )
               ''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
               email VARCHAR(100) NOT NULL
               )
               ''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
               order_date DATE NOT NULL,
               FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
               )
               ''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
                product_id INTEGER NOT NULL,
                quantity INTEGER NOT NULL,
                FOREIGN KEY (order_id) REFERENCES orders(order_id),
                FOREIGN KEY (product_id) REFERENCES products(product_id)
               )
               ''')

In [23]:
# avoid ipython-sql KeyError: 'DEFAULT' incompatibility path; use sqlite3/pandas direct
df_tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type IN ('table', 'view')
  AND name NOT LIKE 'sqlite_%'
ORDER BY 1
""", conn)
df_tables




,name
0,customers
1,order_items
2,orders
3,products


In [20]:
#lets say we made a mistake, if we try to change it just like that without dropping the table,
#we will get an error because we cannot alter a table to add a primary key. So we need to drop the table and
#recreate it with the correct schema.

#our mistake was that our table data 'order_item' was in capital letter, thereby making the rest of the values look untidy
#so we want to correct it now to a small letter like others.

#so now we can drop the table and recreate it with the correct schema.
#create our drop table command
cursor.execute('''
DROP TABLE IF EXISTS products;
''')

cursor.execute('''
DROP TABLE IF EXISTS customers;
''')

cursor.execute('''
DROP TABLE IF EXISTS orders;
''')

cursor.execute('''
DROP TABLE IF EXISTS order_items;
''')



In [24]:
#function to load data from csv to the database table using pandas
def load_data_to_table(csv_path, table_name):
    df = pd.read_csv(csv_path)
    df.to_sql(table_name, conn, if_exists='append', index=False)

In [47]:
load_data_to_table(r'C:\Users\thefa\OneDrive\Desktop\bonga case study\bonga_case_study\sorteddata\order_items - order_items.csv', 'order_items')



In [50]:
df = pd.read_csv(csv_path)
df = df.drop_duplicates(subset=["products"])


NameError: name 'csv_path' is not defined

In [48]:
#lets read our data now
df_order_items = pd.read_sql('SELECT * FROM order_items', conn)
df_order_items.head()

,order_item_id,order_id,product_id,quantity
0,1,1,34,2
1,2,1,653,1
2,3,1,427,3
3,4,2,830,3
4,5,2,886,4
